**ODTSR(Qwen-Image) on Kaggle 2×T4**

In [ ]:
%%javascript
(function(){
  let counter = 0;

  function getFirstLineText(content){
    let lineEl = content.querySelector('.cm-line');
    return lineEl ? lineEl.textContent : '';
  }

  function titleFromLine(line){
    line = (line || '').trim();
    if (line.startsWith('# title:')) return line.replace('# title:', '').trim();
    return line ? 'Code' : '';
  }

  function sync(){
    let editors = document.querySelectorAll('.cm-editor');

    editors.forEach(cm => {
      let content = cm.querySelector('.cm-content');
      if (!content) return;

      let prev = cm.previousElementSibling;
      let hasBtn = prev && prev.classList && prev.classList.contains('cm-toggle-btn');

      let line = getFirstLineText(content);

      if (hasBtn) {
        if (line) {
          let title = titleFromLine(line) || 'Code';
          let collapsed = content.style.display === 'none';
          prev.dataset.title = title;
          prev.textContent = (collapsed ? '▶ ' : '▼ ') + title;
        }
        return;
      }

      if (!line) return;

      let title = titleFromLine(line) || 'Code';
      let btn = document.createElement('button');
      btn.dataset.title = title;
      btn.classList.add('cm-toggle-btn');
      btn.textContent = '▶ ' + title;
      Object.assign(btn.style, {
        marginBottom: '4px',
        background: '#f7f7f7',
        border: '1px solid #ccc',
        padding: '3px 8px',
        borderRadius: '4px',
        cursor: 'pointer',
        fontWeight: 'bold'
      });
      btn.onclick = () => {
        let collapsed = content.style.display === 'none';
        content.style.display = collapsed ? '' : 'none';
        btn.textContent = (collapsed ? '▼ ' : '▶ ') + btn.dataset.title;
      };
      cm.parentNode.insertBefore(btn, cm);
      content.style.display = 'none';
    });

    document.querySelectorAll('.cm-toggle-btn').forEach(btn => {
      let next = btn.nextElementSibling;
      if (!next || !next.classList || !next.classList.contains('cm-editor')) {
        btn.remove();
      }
    });
  }

  sync();

  let scheduled = false;
  function scheduleSync(){
    if (scheduled) return;
    scheduled = true;
    setTimeout(() => { scheduled = false; sync(); }, 50);
  }

  new MutationObserver(scheduleSync)
    .observe(document.body, {childList: true, subtree: true});

  let poll = setInterval(sync, 500);
  setTimeout(() => clearInterval(poll), 15000);
})();


In [ ]:
# title: Install
# Everything here runs once per session: environment setup, patches, and downloading
# all weights (~55 GB). Nothing here depends on the images or on the settings below.
import os, subprocess, torch

# --- GPU check ---
n_gpu = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}, GPU count: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i} -> {p.name}, {p.total_memory/1e9:.1f} GB')
assert n_gpu >= 2, 'You need 2 GPUs! In session settings choose Accelerator: GPU T4 x2'
assert hasattr(torch, 'float8_e4m3fn'), 'torch >= 2.1 is required (already available on Kaggle)'

# --- Clone the repository ---
WORK = '/kaggle/working'
ODTSR_DIR = f'{WORK}/ODTSR'
if not os.path.exists(ODTSR_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RedMediaTech/ODTSR.git', ODTSR_DIR], check=True)

# --- Dependencies ---
# transformers==4.56.0 — the version pinned in the repo's requirements.txt (v5 on Kaggle breaks diffsynth)
%pip -q install einops safetensors sentencepiece protobuf ftfy "transformers==4.56.0" accelerate lightning peft imageio pandas huggingface_hub hf_transfer ipywidgets

# --- Patch for incompatible transformers imports in diffsynth ---
step_path = f'{ODTSR_DIR}/diffsynth/models/stepvideo_text_encoder.py'
if os.path.exists(step_path):
    src = open(step_path).read()
    old_imp = 'from transformers.modeling_utils import PretrainedConfig, PreTrainedModel'
    if old_imp in src:
        src = src.replace(
            old_imp,
            'from transformers.modeling_utils import PreTrainedModel\n'
            'try:\n'
            '    from transformers import PretrainedConfig\n'
            'except ImportError:\n'
            '    from transformers import PreTrainedConfig as PretrainedConfig')
        open(step_path, 'w').write(src)
        print('stepvideo_text_encoder.py: import patch applied')

# --- modelscope stub: shuts down ALL modelscope imports in diffsynth at once ---
# (weights are only downloaded from HuggingFace, modelscope itself is not needed)
import site, sysconfig
site_dir = sysconfig.get_paths()['purelib']
stub_dir = os.path.join(site_dir, 'modelscope')
os.makedirs(os.path.join(stub_dir, 'hub'), exist_ok=True)
with open(os.path.join(stub_dir, '__init__.py'), 'w') as f:
    f.write(
        'def snapshot_download(*a, **k):\n'
        '    raise RuntimeError("modelscope stub: use HuggingFace instead")\n'
        'def dataset_snapshot_download(*a, **k):\n'
        '    raise RuntimeError("modelscope stub: use HuggingFace instead")\n'
    )
with open(os.path.join(stub_dir, 'hub', '__init__.py'), 'w') as f:
    f.write('')
with open(os.path.join(stub_dir, 'hub', 'api.py'), 'w') as f:
    f.write(
        'class HubApi:\n'
        '    def __getattr__(self, name):\n'
        '        raise RuntimeError("modelscope stub: use HuggingFace instead")\n'
    )
print('modelscope stub installed at', stub_dir)

# --- Patch generator.py: enable FP8 storage for DiT weights via an env variable ---
gen_path = f'{ODTSR_DIR}/examples/qwen_image/generator.py'
src = open(gen_path).read()
changed = False
if 'ODTSR_FP8' not in src:
    src = src.replace('use_offload = False', "use_offload = os.environ.get('ODTSR_FP8', '0') == '1'")
    changed = True
if '"seed": 42' in src:
    src = src.replace('"seed": 42', '"seed": int(os.environ.get("ODTSR_SEED", "42"))')
    changed = True
if changed:
    open(gen_path, 'w').write(src)
    print('generator.py: FP8 and seed patches applied')
else:
    print('generator.py: patches already applied')

# --- Patch model_training/train.py: disable the training-dataset import ---
train_path = f'{ODTSR_DIR}/examples/qwen_image/model_training/train.py'
src = open(train_path).read()
ds_imp = 'from diffsynth.extensions.realesrgan.dataset import PairedSROnlineTxtDataset'
if ds_imp in src:
    src = src.replace(ds_imp, 'PairedSROnlineTxtDataset = None  # patched: training dataset not needed for inference')
    open(train_path, 'w').write(src)
    print('train.py: training-dataset import disabled')
else:
    print('train.py: patch already applied')

# --- Patch model_manager.py: convert dtype shard-by-shard while loading ---
mm_path = f'{ODTSR_DIR}/diffsynth/models/model_manager.py'
src = open(mm_path).read()
old_load = 'state_dict.update(load_state_dict(path))'
if old_load in src:
    src = src.replace(old_load, 'state_dict.update(load_state_dict(path, torch_dtype=torch_dtype))')
    open(mm_path, 'w').write(src)
    print('model_manager.py: streaming dtype-conversion patch applied')
else:
    print('model_manager.py: patch already applied')

import sys
for p in (ODTSR_DIR, f'{ODTSR_DIR}/examples/qwen_image'):
    if p not in sys.path:
        sys.path.insert(0, p)

print('\nEnvironment ready. Downloading weights (~55 GB, 15-25 min)...\n')

# --- Download weights (Qwen-Image + ODTSR checkpoint) ---
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files

MODELS_DIR = '/kaggle/tmp/models'
QWEN_DIR   = f'{MODELS_DIR}/Qwen-Image'
os.makedirs(MODELS_DIR, exist_ok=True)

# --- Qwen-Image: only the needed subfolders ---
snapshot_download(
    repo_id='Qwen/Qwen-Image',
    local_dir=QWEN_DIR,
    max_workers=8,
    allow_patterns=[
        'transformer/*.safetensors',
        'text_encoder/*.safetensors',
        'vae/diffusion_pytorch_model.safetensors',
        'tokenizer/*',
        '*.json',
    ],
)
print('Qwen-Image downloaded.')

# --- ODTSR checkpoint (DualLoRA generator) ---
# In the double8fun/ODTSR repo:
#   weight.pth              (2.46 GB) — the GENERATOR, this is the one we need
#   net_dis_iter_5001.pth   (2.84 GB) — discriminator, NOT needed for inference
# The old code picked pth_files[0] and downloaded the discriminator — hence the blur.
ODTSR_REPO = 'double8fun/ODTSR'
GEN_CKPT_NAME = 'weight.pth'

repo_files = list_repo_files(ODTSR_REPO)
assert GEN_CKPT_NAME in repo_files, (
    f'{GEN_CKPT_NAME} not found in {ODTSR_REPO}. Available .pth files: '
    f'{[f for f in repo_files if f.endswith(".pth")]}'
)
CKPT_PATH = hf_hub_download(ODTSR_REPO, GEN_CKPT_NAME, local_dir=MODELS_DIR)
print(f'ODTSR generator checkpoint: {CKPT_PATH} ({os.path.getsize(CKPT_PATH)/1e9:.2f} GB)')
assert 'net_dis' not in os.path.basename(CKPT_PATH), 'Downloaded the discriminator instead of the generator!'

import shutil
total, used, free = shutil.disk_usage('/kaggle/tmp')
print(f'Disk /kaggle/tmp: used {used/1e9:.0f} GB, free {free/1e9:.0f} GB')

print('\nInstall complete.')

In [ ]:
# title: Upload Images
# Upload files with the button below, or place images into /kaggle/working/inputs
# (datasets under /kaggle/input are also picked up if you point the extra-folder field at them).
# Running this cell, or uploading a new batch, wipes any previous input/output files first.
import os, shutil, ipywidgets as W
from IPython.display import display

INPUT_DIR = '/kaggle/working/inputs'
OUTPUT_DIR = '/kaggle/working/outputs'

def _clear_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

# Wipe old inputs/outputs every time this cell runs
_clear_dir(INPUT_DIR)
_clear_dir(OUTPUT_DIR)

upload_w = W.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=True, description='Upload files')
extra_dir_w = W.Text(value='', placeholder='/kaggle/input/my-dataset (optional)',
                     description='Extra folder:', style={'description_width': '90px'}, layout=W.Layout(width='560px'))
status_w = W.HTML()

def _save_uploads(*_):
    # A new batch was uploaded -> drop everything left over from the previous one
    _clear_dir(INPUT_DIR)
    _clear_dir(OUTPUT_DIR)
    files = upload_w.value
    items = files.values() if isinstance(files, dict) else files  # ipywidgets 7/8 compatibility
    saved = 0
    for f in items:
        name = f['name'] if isinstance(f, dict) else f.name
        content = f['content'] if isinstance(f, dict) else f.content
        content = content.tobytes() if hasattr(content, 'tobytes') else bytes(content)
        with open(os.path.join(INPUT_DIR, name), 'wb') as out:
            out.write(content)
        saved += 1
    _refresh(saved)

def _refresh(just_saved=0):
    exts = ('.png', '.jpg', '.jpeg', '.webp')
    listing = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith(exts))
    msg = f'<b>Just saved: {just_saved}.</b> ' if just_saved else ''
    status_w.value = msg + f'Queued for upscale ({len(listing)}): ' + ', '.join(listing[:20])

upload_w.observe(_save_uploads, 'value')
_refresh()
display(W.VBox([upload_w, extra_dir_w, status_w]))

In [ ]:
# title: Config
# Choose the quantization mode and upscale parameters.
# All options are sized to fit into 2x15 GB VRAM and 29 GB RAM.
# Defaults below are pre-tuned for a good sharpness/speed balance — adjust only if needed.
import ipywidgets as W
from IPython.display import display, HTML

QUANT_OPTIONS = {
    'FP8 — balanced across 2 GPUs (recommended)': 'fp8_balanced',
    'FP8 — GPU0 + CPU offload (slower, fallback)': 'fp8_gpu0_cpu',
    'BF16 — no quantization (does NOT fit on 2xT4)': 'bf16',
}

quant_w = W.RadioButtons(
    options=list(QUANT_OPTIONS.keys()),
    value=list(QUANT_OPTIONS.keys())[0],
    description='Quantization:',
    style={'description_width': '120px'},
    layout=W.Layout(width='680px'),
)

vram_info = W.HTML()
def _update_info(*_):
    mode = QUANT_OPTIONS[quant_w.value]
    rows = {
        'fp8_balanced': ('~12 GB', '~11 GB', '~24 GB (peak while loading)', 'full speed'),
        'fp8_gpu0_cpu': ('~13 GB', '~0 GB', '~26 GB (block offload)', '3-5x slower'),
        'bf16':         ('41 GB — OOM!', '—', 'OOM', '—'),
    }[mode]
    color = '#b00020' if mode == 'bf16' else '#1a7f37'
    vram_info.value = (
        f'<table style="border-collapse:collapse;font-size:13px">'
        f'<tr><th style="padding:2px 10px;text-align:left">GPU 0</th>'
        f'<th style="padding:2px 10px;text-align:left">GPU 1</th>'
        f'<th style="padding:2px 10px;text-align:left">RAM</th>'
        f'<th style="padding:2px 10px;text-align:left">Speed</th></tr>'
        f'<tr style="color:{color}">' + ''.join(f'<td style="padding:2px 10px">{v}</td>' for v in rows) + '</tr></table>'
    )
quant_w.observe(_update_info, 'value'); _update_info()

# scale default 2.0x — the model was primarily trained for x2 and stays sharpest there;
# push it to x3-x4 only if you specifically need a larger output.
scale_w  = W.FloatSlider(value=2.0, min=1.0, max=4.0, step=0.5, description='Scale (x):',
                         style={'description_width': '120px'}, layout=W.Layout(width='500px'))
align_w  = W.Dropdown(options=[('Wavelet (recommended)', 'wavelet'), ('AdaIN', 'adain'), ('No color fix', 'no')],
                      value='wavelet', description='Color fix:',
                      style={'description_width': '120px'}, layout=W.Layout(width='400px'))

prompt_w = W.Textarea(
    value='raw photo, extremely detailed, sharp focus, realistic texture, high contrast, '
          '8k UHD, photorealistic',
    description='Prompt:', style={'description_width': '120px'},
    layout=W.Layout(width='680px', height='70px'))

neg_prompt_w = W.Textarea(
    value='',
    description='Negative:', style={'description_width': '120px'},
    layout=W.Layout(width='680px', height='70px'))

seed_w   = W.IntText(value=42, description='Seed:', style={'description_width': '120px'}, layout=W.Layout(width='260px'))
tile_info = W.HTML('<span style="font-size:13px;color:#555">Tile: 64 latent px (512 px), stride 48 — fixed, matches training.</span>')

# fidelity < 1.0 adds noise to the LQ latents so the model "fills in" more detail;
# 0.85 is a good default that sharpens results without drifting far from the source.
fidelity_w = W.FloatSlider(value=1, min=0.0, max=1.0, step=0.05, description='Fidelity:',
                           style={'description_width': '120px'}, layout=W.Layout(width='500px'))
# cfg > 1 doubles inference time but follows the prompt more strongly and reduces smudging;
# 1.5 is a solid default sharpness/speed compromise.
cfg_w = W.FloatSlider(value=1, min=1.0, max=4.0, step=0.5, description='CFG:',
                      style={'description_width': '120px'}, layout=W.Layout(width='500px'))
sharp_info = W.HTML('<span style="font-size:13px;color:#555">Result too smudged? Try Fidelity 0.7-0.9 (more "filled-in" detail) '
                    'and/or CFG 1.5-2.0 (stronger prompt following; CFG > 1 doubles inference time). '
                    'Scale x2 instead of x4 also helps, since the model was mainly trained at x2.</span>')

encode_dev_w = W.RadioButtons(
    options=['CPU (reliable, ~5-10 min)', 'GPU (faster, ~1-2 min; ONLY before Run, experimental)'],
    value='CPU (reliable, ~5-10 min)', description='Encoder:',
    style={'description_width': '120px'}, layout=W.Layout(width='680px'))

dual_stream_w = W.Checkbox(
    value=False, indent=False,
    description='Tile pipelining: both GPUs work simultaneously (experimental; +1-2 GB VRAM per card, up to ~1.7x speedup)')

tab = W.Tab()
tab.children = [
    W.VBox([scale_w, align_w, prompt_w, neg_prompt_w, seed_w, tile_info]),
    W.VBox([fidelity_w, cfg_w, sharp_info]),
    W.VBox([quant_w, vram_info, dual_stream_w, encode_dev_w]),
]
tab.set_title(0, '📝 Prompts & Scale')
tab.set_title(1, '⚙️ Sampling')
tab.set_title(2, '🖥️ GPU & Quantization')

display(tab)


In [ ]:
# title: Run
# Runs the whole pipeline for every image currently in the queue: encodes the prompt
# (only if it changed), loads the model onto the GPUs (only the first time this cell
# runs), then upscales. No before/after preview here — see the Visualization cell for that.
import os, gc, json, types, glob, time, subprocess, torch
from PIL import Image

# ============================================================
# 1) Encode prompt (skipped if already cached)
# ============================================================
PROMPT_CACHE_PATH = '/kaggle/tmp/prompt_cache.pt'
prompts_to_encode = sorted({prompt_w.value.strip(), neg_prompt_w.value.strip(), ''})

cache = torch.load(PROMPT_CACHE_PATH, map_location='cpu') if os.path.exists(PROMPT_CACHE_PATH) else {}
missing = [p for p in prompts_to_encode if p not in cache]

if not missing:
    print('Prompt cache: up to date ->', list(cache.keys()))
else:
    enc_mode = 'gpu' if encode_dev_w.value.startswith('GPU') else 'cpu'
    if enc_mode == 'gpu' and 'model' in globals():
        print('The model is already loaded and using VRAM — switching to CPU encoding.')
        enc_mode = 'cpu'
    print(f'Encoding prompts ({enc_mode.upper()}, 7B encoder):', missing)
    encoder_script = r'''
import sys, os, json, torch
sys.path.insert(0, os.environ['ODTSR_DIR'])
from diffsynth.pipelines.qwen_image import QwenImagePipeline, ModelConfig

qwen = os.environ['QWEN_DIR']
te_shards = [f"{qwen}/text_encoder/model-{i:05d}-of-00004.safetensors" for i in range(1, 5)]

pipe = QwenImagePipeline.from_pretrained(
    torch_dtype=torch.bfloat16, device='cpu',
    model_configs=[ModelConfig(path=te_shards)],
    tokenizer_config=ModelConfig(f"{qwen}/tokenizer"),
)
unit = pipe.units[-1]  # QwenImageUnit_PromptEmbedder

if os.environ.get('ODTSR_ENCODE_DEVICE', 'cpu') == 'gpu':
    te = pipe.text_encoder
    best = None
    for _n, _m in te.named_modules():
        if isinstance(_m, torch.nn.ModuleList) and (best is None or len(_m) > len(best)):
            best = _m
    split = (len(best) + 1) // 2
    for i, l in enumerate(best):
        l.to('cuda:0' if i < split else 'cuda:1')
    for m in te.modules():
        for p in list(m.parameters(recurse=False)):
            if p.device.type == 'cpu':
                p.data = p.data.to('cuda:0')
        for bn, b in list(m.named_buffers(recurse=False)):
            if b is not None and b.device.type == 'cpu':
                setattr(m, bn, b.to('cuda:0'))
    def _mv(obj, device):
        if torch.is_tensor(obj): return obj.to(device)
        if isinstance(obj, tuple): return tuple(_mv(o, device) for o in obj)
        if isinstance(obj, list): return [_mv(o, device) for o in obj]
        if isinstance(obj, dict): return {k: _mv(v, device) for k, v in obj.items()}
        return obj
    def _mk(dev):
        def hook(module, args, kwargs):
            return _mv(args, dev), _mv(kwargs, dev)
        return hook
    for m in te.modules():
        try:
            dev = next(m.parameters()).device
        except StopIteration:
            continue
        m.register_forward_pre_hook(_mk(dev), with_kwargs=True)
    pipe.device = 'cuda:0'
    print(f'text encoder split across 2 GPUs: {split}/{len(best) - split} layers', flush=True)

cache_path = os.environ['PROMPT_CACHE_PATH']
cache = torch.load(cache_path, map_location='cpu') if os.path.exists(cache_path) else {}
for p in json.loads(os.environ['PROMPTS_JSON']):
    print('encode:', repr(p), flush=True)
    with torch.no_grad():
        out = unit.process(pipe, prompt=p)
    cache[p] = {k: (v.detach().to('cpu') if torch.is_tensor(v) else v) for k, v in out.items()}
torch.save(cache, cache_path)
print('saved:', list(cache.keys()), flush=True)
'''
    script_path = '/kaggle/tmp/encode_prompts.py'
    open(script_path, 'w').write(encoder_script)
    env = dict(os.environ,
               ODTSR_DIR=ODTSR_DIR, QWEN_DIR=QWEN_DIR,
               PROMPT_CACHE_PATH=PROMPT_CACHE_PATH,
               PROMPTS_JSON=json.dumps(missing),
               ODTSR_ENCODE_DEVICE=enc_mode,
               CUDA_VISIBLE_DEVICES=('0,1' if enc_mode == 'gpu' else ''))
    r = subprocess.run(['python', script_path], env=env, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError('Prompt encoding failed — see the log above. If GPU mode was selected, try CPU in Config.')

PROMPT_CACHE = torch.load(PROMPT_CACHE_PATH, map_location='cpu')

# ============================================================
# 2) Load model onto the GPUs — only once per session
# ============================================================
if 'model' not in globals():
    QUANT_MODE = QUANT_OPTIONS[quant_w.value]
    assert QUANT_MODE != 'bf16', 'BF16 (41 GB) does not fit on 2xT4 — choose FP8 in Config.'
    os.environ['ODTSR_FP8'] = '1'

    os.chdir(f'{ODTSR_DIR}/examples/qwen_image')
    from generator import Generator

    transformer_shards = [f'{QWEN_DIR}/transformer/diffusion_pytorch_model-{i:05d}-of-00009.safetensors' for i in range(1, 10)]
    weights_json = json.dumps([transformer_shards, f'{QWEN_DIR}/vae/diffusion_pytorch_model.safetensors'])

    print('Loading DiT (FP8) + VAE on CPU... (~5-10 min, only happens once)')
    model = Generator(
        torch_dtype=torch.bfloat16,
        pretrained_weights=weights_json,
        tokenizer_path=f'{QWEN_DIR}/tokenizer',
        learning_rate=0,
        use_gradient_checkpointing=False,
        pretrained_ckpt_path_gen=None,
    )
    model.pipe.requires_grad_(False)
    model.eval()
    gc.collect()

    from model_training.train import LinearFP8Wrapper
    n_fixed = 0
    for m in model.pipe.dit.modules():
        if isinstance(m, (torch.nn.Linear, LinearFP8Wrapper)):
            continue
        for p in m.parameters(recurse=False):
            if p.dtype == torch.float8_e4m3fn:
                p.data = p.data.to(torch.bfloat16); n_fixed += 1
        for bname, buf in list(m.named_buffers(recurse=False)):
            if buf is not None and buf.dtype == torch.float8_e4m3fn:
                setattr(m, bname, buf.to(torch.bfloat16)); n_fixed += 1
    print(f'fp8 -> bf16 for non-Linear weights: {n_fixed} tensors')

    dit = model.pipe.dit
    blocks = dit.transformer_blocks
    n = len(blocks)
    print(f'DiT: {n} blocks')

    if QUANT_MODE == 'fp8_balanced':
        split = n // 2
        block_devices = ['cuda:0' if i < split else 'cuda:1' for i in range(n)]
        tail_device = 'cuda:1'
    else:  # fp8_gpu0_cpu
        on_gpu = 34
        block_devices = ['cuda:0' if i < on_gpu else 'cpu' for i in range(n)]
        tail_device = 'cuda:0'

    print('Moving blocks to their devices...')
    for name in ('pos_embed', 'time_text_embed', 'txt_norm', 'img_in', 'txt_in'):
        getattr(dit, name).to('cuda:0')
    for i, b in enumerate(blocks):
        b.to(block_devices[i])
        if i % 10 == 9:
            gc.collect(); torch.cuda.empty_cache()
    dit.norm_out.to(tail_device)
    dit.proj_out.to(tail_device)

    model.pipe.vae.to('cuda:0')
    model.pipe.new_vae.to('cuda:0')
    model.device = torch.device('cuda:0')
    model.pipe.device = 'cuda:0'
    gc.collect(); torch.cuda.empty_cache()

    def _move(obj, device):
        if torch.is_tensor(obj):
            return obj.to(device)
        if isinstance(obj, tuple):
            return tuple(_move(o, device) for o in obj)
        if isinstance(obj, list):
            return [_move(o, device) for o in obj]
        if isinstance(obj, dict):
            return {k: _move(v, device) for k, v in obj.items()}
        return obj

    def _make_pre_hook(device):
        def hook(module, args, kwargs):
            return _move(args, device), _move(kwargs, device)
        return hook

    hooked = list(blocks) + [dit.norm_out, dit.proj_out, dit.img_in, dit.txt_in, dit.txt_norm, dit.time_text_embed]
    for m in hooked:
        try:
            dev = next(m.parameters()).device
        except StopIteration:
            continue
        m.register_forward_pre_hook(_make_pre_hook(dev), with_kwargs=True)
    dit.proj_out.register_forward_hook(lambda mod, args, out: _move(out, 'cuda:0'))

    print('Loading the ODTSR checkpoint...')
    assert 'net_dis' not in os.path.basename(CKPT_PATH), (
        f'CKPT_PATH points to the discriminator: {CKPT_PATH}. '
        'Re-run the Install cell — it must download weight.pth.'
    )
    state_dict = torch.load(CKPT_PATH, map_location='cpu')
    # In case of a training wrapper like {'state_dict': {...}}
    if isinstance(state_dict, dict) and 'state_dict' in state_dict and not torch.is_tensor(state_dict['state_dict']):
        state_dict = state_dict['state_dict']

    missing_k, unexpected_k = model.load_state_dict(state_dict, strict=False)
    print(f'load_state_dict: missing={len(missing_k)}, unexpected={len(unexpected_k)}')

    # Hard validation: the checkpoint MUST actually land in the model
    assert len(unexpected_k) == 0, (
        f'{len(unexpected_k)} checkpoint keys did not match the model, examples: {unexpected_k[:5]}'
    )
    lora_missing = [k for k in missing_k if 'lora' in k.lower()]
    assert len(lora_missing) == 0, (
        f'LoRA weights were NOT loaded ({len(lora_missing)} keys), examples: {lora_missing[:5]}'
    )
    n_lora_loaded = sum(1 for k in state_dict if 'lora' in k.lower())
    print(f'OK: generator checkpoint applied, LoRA tensors loaded: {n_lora_loaded}')
    del state_dict
    gc.collect(); torch.cuda.empty_cache()

    for i in range(2):
        a = torch.cuda.memory_allocated(i) / 1e9
        print(f'GPU {i}: {a:.1f} GB used')
    print('Model loaded.\n')
else:
    print('Model already loaded — skipping reload.\n')

# prompt embedder always re-bound to the (possibly refreshed) cache
prompt_unit = model.pipe.units[-1]
def cached_process(self, pipe, prompt=None, **kwargs):
    key = (prompt or '').strip()
    if key not in PROMPT_CACHE:
        raise RuntimeError(f'Prompt {key!r} is not cached — re-run this cell.')
    return {k: (v.to(pipe.device) if torch.is_tensor(v) else v) for k, v in PROMPT_CACHE[key].items()}
prompt_unit.process = types.MethodType(cached_process, prompt_unit)

# ============================================================
# 3) Tile pipelining (Config option)
# ============================================================
import diffsynth.pipelines.qwen_image as qi_mod
if '_ORIG_TILED_MODEL_FN' not in globals():
    _ORIG_TILED_MODEL_FN = qi_mod.tiled_model_fn_qwen_image

if dual_stream_w.value:
    from concurrent.futures import ThreadPoolExecutor
    from einops import rearrange as _rearr
    from diffsynth.models.tiler import TileWorker as _TW

    class _PipelinedTileWorker(_TW):
        def tiled_inference(self, forward_fn, model_input, tile_batch_size,
                            inference_device, inference_dtype, tile_device, tile_dtype):
            tile_num = model_input.shape[-1]
            def _run(tile_id):
                x = model_input[:, :, :, :, tile_id:tile_id + 1].to(device=inference_device, dtype=inference_dtype)
                x = _rearr(x, 'b c h w n -> (n b) c h w')
                y = forward_fn(x)
                y = _rearr(y, '(n b) c h w -> b c h w n', n=1)
                return y.to(device=tile_device, dtype=tile_dtype)
            with ThreadPoolExecutor(max_workers=2) as ex:
                outs = list(ex.map(_run, range(tile_num)))
            return torch.concat(outs, dim=-1)

    def _pipelined_tiled_model_fn(dit, latents, condition_latents, timestep,
                                  prompt_emb, prompt_emb_mask, tile_size, tile_stride):
        def chunk_function(x):
            split_size = x.size(1) // 2
            hidden_part, lr_part = torch.split(x, split_size, dim=1)
            return qi_mod.model_fn_qwen_image(
                dit, hidden_part, lr_part, timestep, prompt_emb, prompt_emb_mask,
                hidden_part.shape[2] * 8, hidden_part.shape[3] * 8)
        return _PipelinedTileWorker().tiled_forward(
            chunk_function, torch.concat([latents, condition_latents], dim=1),
            tile_size, tile_stride, tile_device=latents.device, tile_dtype=latents.dtype)

    qi_mod.tiled_model_fn_qwen_image = _pipelined_tiled_model_fn
    print('Tile pipelining: ON (both GPUs simultaneously). If you hit OOM, uncheck it in Config.')
else:
    qi_mod.tiled_model_fn_qwen_image = _ORIG_TILED_MODEL_FN

# ============================================================
# 4) Upscale every queued image (no preview — see Visualization cell)
# ============================================================
TILESIZE = 64
TILE_STRIDE = TILESIZE - TILESIZE // 4  # 48, matches training

def adaptive_pad(img, tilesize, stride):
    w, h = img.size
    pad_h = (tilesize - h) if h <= tilesize else ((h - tilesize + stride - 1) // stride) * stride + tilesize - h
    pad_w = (tilesize - w) if w <= tilesize else ((w - tilesize + stride - 1) // stride) * stride + tilesize - w
    new_img = Image.new(img.mode, (w + pad_w, h + pad_h), color=0)
    new_img.paste(img, (0, 0))
    return new_img

scale = float(scale_w.value)
prompt = prompt_w.value.strip()
negative_prompt = neg_prompt_w.value.strip()
align_method = align_w.value
fidelity = float(fidelity_w.value)
cfg_scale = float(cfg_w.value)
torch.manual_seed(int(seed_w.value))
os.environ['ODTSR_SEED'] = str(int(seed_w.value))

exts = ('*.png', '*.jpg', '*.jpeg', '*.webp')
paths = []
for e in exts:
    paths += glob.glob(os.path.join(INPUT_DIR, e))
    if extra_dir_w.value.strip():
        paths += glob.glob(os.path.join(extra_dir_w.value.strip(), '**', e), recursive=True)
paths = sorted(set(paths))
assert paths, 'No images found! Upload some in Upload Images.'
print(f'Images: {len(paths)}, scale x{scale}, fidelity={fidelity}, cfg={cfg_scale}, prompt: {prompt!r}, negative: {negative_prompt!r}\n')

from wavelet_color_fix import adain_color_fix, wavelet_color_fix

RESULTS = []  # (input_path, output_path) pairs for this run — used by Visualization

for idx, img_path in enumerate(paths):
    name = os.path.splitext(os.path.basename(img_path))[0]
    res_path = os.path.join(OUTPUT_DIR, f'{name}_x{scale:g}.png')
    print(f'[{idx+1}/{len(paths)}] {os.path.basename(img_path)}', end=' ')
    t0 = time.perf_counter()

    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    w_dst, h_dst = round(w * scale), round(h * scale)
    upsampled = img.resize((w_dst, h_dst), Image.BICUBIC)
    padded = adaptive_pad(upsampled, tilesize=TILESIZE * 8, stride=TILE_STRIDE * 8)

    with torch.no_grad():
        res = model.infer(
            prompt=prompt,
            negative_prompt=negative_prompt,
            condition_image=padded,
            cfg_scale=cfg_scale,
            fidelity=fidelity,
            tiled=True,
            tile_size=TILESIZE,
            tile_stride=TILE_STRIDE,
        )

    res = res.crop((0, 0, w_dst, h_dst))
    if align_method == 'wavelet':
        res = wavelet_color_fix(res, upsampled)
    elif align_method == 'adain':
        res = adain_color_fix(res, upsampled)
    res.save(res_path)
    torch.cuda.empty_cache()

    RESULTS.append((img_path, res_path))
    print(f'-> {res_path} ({time.perf_counter()-t0:.1f} s)')

print(f'\nDone. {len(RESULTS)} image(s) saved to {OUTPUT_DIR}. Open the Visualization cell to compare before/after.')

In [ ]:
# title: Visualization
import os, base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

if 'RESULTS' not in globals() or not RESULTS:
    raise RuntimeError('No results found — run the Run cell first.')

# Extensions the browser's canvas.toDataURL can encode directly.
# Anything else (bmp, tiff, ...) is saved as PNG on download.
CANVAS_MIME = {
    '.png': 'image/png',
    '.jpg': 'image/jpeg',
    '.jpeg': 'image/jpeg',
    '.webp': 'image/webp',
}

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format='PNG')
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def fit_box(width, height, max_width, target_height=None):
    # Only used to size the on-screen comparison box; the embedded image data
    # itself stays full resolution so the download matches the real upscale.
    if width > max_width:
        height = int(height * max_width / width)
        width = max_width
    if target_height is not None and height != target_height:
        width = int(width * target_height / height)
        height = target_height
    return width, height

for idx, (img_path, res_path) in enumerate(RESULTS):
    try:
        filename = os.path.basename(img_path)
        out_filename = os.path.basename(res_path)

        image_before = PIL.Image.open(img_path)
        image_after = PIL.Image.open(res_path)

        if image_before.mode != 'RGB':
            image_before = image_before.convert('RGB')
        if image_after.mode != 'RGB':
            image_after = image_after.convert('RGB')

        # Display-only box size (full-res pixel data is embedded either way)
        max_width = min(image_after.size[0], 700)
        display_w, display_h = fit_box(*image_after.size, max_width)

        before_base64 = image_to_base64(image_before)
        after_base64 = image_to_base64(image_after)

        # Use the result's original format for the download if the browser can encode it via
        # canvas, otherwise fall back to PNG.
        out_ext = os.path.splitext(res_path)[1].lower()
        download_mime = CANVAS_MIME.get(out_ext, 'image/png')
        download_ext = out_ext if out_ext in CANVAS_MIME else '.png'

        uid = f"cmp{idx}"
        base_name = os.path.splitext(filename)[0]

        html_code = f"""
        <style>
            .download-btn-{uid} {{
                background-color: #28a745;
                color: white;
                border: none;
                border-radius: 4px;
                padding: 5px 10px;
                cursor: pointer;
                display: flex;
                align-items: center;
                gap: 6px;
                font-family: sans-serif;
                font-size: 12px;
                font-weight: bold;
                flex-shrink: 0;
                transition: background-color 0.15s ease-in-out;
            }}
            .download-btn-{uid}:hover {{
                background-color: #218838;
            }}
            .download-btn-{uid}:active {{
                background-color: #1e7e34;
            }}
        </style>
        <div style="width: {display_w}px; margin-bottom: 30px;">
            <div style="margin-bottom:6px; font-family: sans-serif; font-size: 13px; font-weight: bold;">
                {filename} &rarr; {out_filename}
            </div>
            <div id="{uid}" style="position: relative; width: 100%; height: {display_h}px;">
                <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                    <img class="img-before" src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                    <div class="clip-wrap" style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                        <canvas class="canvas-after" style="position: absolute; width: 100%; height: 100%; opacity: 1;"></canvas>
                    </div>
                </div>
                <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                    <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                </div>
                <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
            </div>

            <div style="display: flex; flex-direction: column; gap: 8px; margin-top: 8px; font-family: sans-serif; font-size: 13px; color: inherit; min-width: 360px;">
                <div style="display: flex; align-items: center; gap: 10px;">
                    <span style="white-space: nowrap; flex-shrink: 0; min-width: 90px;">Blend original:</span>
                    <input type="range" class="blend-slider" min="0" max="100" value="100" step="1" style="flex: 1;">
                    <span class="blend-value" style="min-width: 45px; text-align: right; flex-shrink: 0;">100%</span>
                    <div style="width: 100px; flex-shrink: 0;"></div>
                </div>

                <div style="display: flex; align-items: center; gap: 10px;">
                    <span style="white-space: nowrap; flex-shrink: 0; min-width: 90px;">Sharpness:</span>
                    <input type="range" class="sharpness-slider" min="0.0" max="7.0" value="0.0" step="0.1" style="flex: 1;">
                    <span class="sharpness-value" style="min-width: 45px; text-align: right; flex-shrink: 0;">0.0</span>
                    <button class="download-btn-{uid}" title="Download blended and sharpened image ({download_ext}, full resolution)" style="width: 100px;">
                        <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" style="display: inline-block;">
                            <path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path>
                            <polyline points="7 10 12 15 17 10"></polyline>
                            <line x1="12" y1="15" x2="12" y2="3"></line>
                        </svg>
                        Download
                    </button>
                </div>
            </div>
        </div>
        <script>
            (function() {{
                const root = document.getElementById('{uid}');
                const slider = root.querySelector('.slider');
                const container = root.querySelector('div:nth-child(2)');
                const clipDiv = root.querySelector('.clip-wrap');
                let isDragging = false;

                slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                document.addEventListener('mouseup', () => {{ isDragging = false; }});
                document.addEventListener('mousemove', (e) => {{
                    if (!isDragging) return;
                    const rect = container.getBoundingClientRect();
                    let x = e.clientX - rect.left;
                    if (x < 0) x = 0;
                    if (x > rect.width) x = rect.width;
                    const percentage = (x / rect.width) * 100;
                    slider.style.left = percentage + '%';
                    clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                }});

                slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                document.addEventListener('touchend', () => {{ isDragging = false; }});
                document.addEventListener('touchmove', (e) => {{
                    if (!isDragging) return;
                    const rect = container.getBoundingClientRect();
                    let x = e.touches[0].clientX - rect.left;
                    if (x < 0) x = 0;
                    if (x > rect.width) x = rect.width;
                    const percentage = (x / rect.width) * 100;
                    slider.style.left = percentage + '%';
                    clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                }});
            }})();

            (function() {{
                const root = document.getElementById('{uid}');
                const wrapper = root.parentElement;
                const blendSlider = wrapper.querySelector('.blend-slider');
                const blendValue = wrapper.querySelector('.blend-value');
                const sharpnessSlider = wrapper.querySelector('.sharpness-slider');
                const sharpnessValue = wrapper.querySelector('.sharpness-value');

                const canvasAfter = root.querySelector('.canvas-after');
                const downloadBtn = wrapper.querySelector('.download-btn-{uid}');

                const downloadMime = "{download_mime}";
                const downloadExt = "{download_ext}";

                const imgBeforeSource = new Image();
                imgBeforeSource.src = "data:image/png;base64,{before_base64}";

                const imgAfterSource = new Image();
                imgAfterSource.src = "data:image/png;base64,{after_base64}";

                let originalImgData = null;
                let smoothedPixels = null;
                let ctx = null;

                function getSmoothedPixels(imgData) {{
                    const w = imgData.width;
                    const h = imgData.height;
                    const src = imgData.data;
                    const dst = new Uint8ClampedArray(src.length);
                    dst.set(src);

                    const kernel = [
                        1, 1, 1,
                        1, 5, 1,
                        1, 1, 1
                    ];
                    const divisor = 13;

                    for (let y = 1; y < h - 1; y++) {{
                        for (let x = 1; x < w - 1; x++) {{
                            let r = 0, g = 0, b = 0;
                            for (let ky = -1; ky <= 1; ky++) {{
                                const rowOffset = (y + ky) * w * 4;
                                for (let kx = -1; kx <= 1; kx++) {{
                                    const idx = rowOffset + (x + kx) * 4;
                                    const weight = kernel[(ky + 1) * 3 + (kx + 1)];
                                    r += src[idx] * weight;
                                    g += src[idx + 1] * weight;
                                    b += src[idx + 2] * weight;
                                }}
                            }}
                            const dstIdx = (y * w + x) * 4;
                            dst[dstIdx]     = Math.min(255, Math.max(0, r / divisor));
                            dst[dstIdx + 1] = Math.min(255, Math.max(0, g / divisor));
                            dst[dstIdx + 2] = Math.min(255, Math.max(0, b / divisor));
                        }}
                    }}
                    return dst;
                }}

                function applySharpness(factor) {{
                    if (!originalImgData || !smoothedPixels) return;

                    const w = canvasAfter.width;
                    const h = canvasAfter.height;

                    if (factor === 1.0) {{
                        ctx.putImageData(originalImgData, 0, 0);
                        return;
                    }}

                    const src = originalImgData.data;
                    const smoothed = smoothedPixels;

                    const outputData = ctx.createImageData(w, h);
                    const dst = outputData.data;

                    for (let i = 0; i < src.length; i += 4) {{
                        dst[i]     = Math.min(255, Math.max(0, smoothed[i] * (1 - factor) + src[i] * factor));
                        dst[i + 1] = Math.min(255, Math.max(0, smoothed[i + 1] * (1 - factor) + src[i + 1] * factor));
                        dst[i + 2] = Math.min(255, Math.max(0, smoothed[i + 2] * (1 - factor) + src[i + 2] * factor));
                        dst[i + 3] = src[i + 3];
                    }}
                    ctx.putImageData(outputData, 0, 0);
                }}

                imgAfterSource.onload = () => {{
                    // Backing-store resolution = the real upscaled image (imgAfterSource is
                    // full-res); CSS width/height:100% on the canvas scale it down for the
                    // compact on-screen preview only. This is what makes the download full-res.
                    canvasAfter.width = imgAfterSource.naturalWidth;
                    canvasAfter.height = imgAfterSource.naturalHeight;
                    ctx = canvasAfter.getContext('2d');

                    ctx.drawImage(imgAfterSource, 0, 0);
                    originalImgData = ctx.getImageData(0, 0, canvasAfter.width, canvasAfter.height);

                    smoothedPixels = getSmoothedPixels(originalImgData);

                    applySharpness(0.0);
                }};

                sharpnessSlider.addEventListener('input', (e) => {{
                    const val = parseFloat(e.target.value);
                    sharpnessValue.textContent = val.toFixed(1);
                    applySharpness(val);
                }});

                blendSlider.addEventListener('input', (e) => {{
                    const val = e.target.value;
                    blendValue.textContent = val + '%';
                    canvasAfter.style.opacity = val / 100;
                }});

                downloadBtn.addEventListener('click', () => {{
                    // w/h come from canvasAfter's backing store, i.e. the full upscaled resolution
                    const canvas = document.createElement('canvas');
                    const w = canvasAfter.width;
                    const h = canvasAfter.height;
                    canvas.width = w;
                    canvas.height = h;

                    const downloadCtx = canvas.getContext('2d');

                    downloadCtx.drawImage(imgBeforeSource, 0, 0, w, h);

                    const opacity = parseFloat(blendSlider.value) / 100;
                    downloadCtx.globalAlpha = opacity;
                    downloadCtx.drawImage(canvasAfter, 0, 0, w, h);

                    const link = document.createElement('a');
                    link.download = 'blended_{base_name}' + downloadExt;
                    link.href = canvas.toDataURL(downloadMime, 0.95);
                    link.click();
                }});
            }})();
        </script>
        """

        display(HTML(html_code))

    except Exception as e:
        print(f'Error processing {img_path} and {res_path}: {e}')

In [ ]:
# title: Download Results
# One green button. Click -> (zip if several files) -> encode -> the download
# starts automatically in the browser. No file hosts, no extra buttons.
# NOTE: the green color is forced via btn.style.button_color because Kaggle's
# theme overrides ipywidgets' standard button_style colors (button looked black).
import os, base64, zipfile, mimetypes, time
import ipywidgets as widgets
from IPython.display import HTML, display

CHUNK_MB = 50  # max size of a single auto-download; bigger sets are split into parts

results = sorted(f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith('.png'))
assert results, 'No results — run the Run cell first.'

SPINNER = ('<style>.odtsr-spin{display:inline-block;width:16px;height:16px;margin-right:8px;'
           'border:3px solid #c8e6c9;border-top-color:#4CAF50;border-radius:50%;'
           'animation:odtsr-rot .8s linear infinite;vertical-align:middle}'
           '@keyframes odtsr-rot{to{transform:rotate(360deg)}}</style>'
           '<div style="font-family:sans-serif;font-size:14px;margin:6px 0">'
           '<span class="odtsr-spin"></span>MSG</div>')

def ok_html(msg):
    return f'<div style="font-family:sans-serif;font-size:14px;color:#2E7D32;margin:6px 0">{msg}</div>'

def err_html(msg):
    return f'<div style="font-family:sans-serif;font-size:14px;color:#B71C1C;margin:6px 0">{msg}</div>'

total_mb = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in results) / 1e6
print(f'{len(results)} image(s), ~{total_mb:.1f} MB total.')
print('(Also always available with no clicks: Output panel on the right -> /kaggle/working,')
print(' or press "Save Version" and download from the notebook\'s Output tab.)')

# ---------- UI: force the green color, Kaggle's theme ignores button_style ----------
btn = widgets.Button(description='Download', layout=widgets.Layout(width='320px', height='38px'))
btn.style.button_color = '#4CAF50'
try:
    btn.style.text_color = 'white'   # ipywidgets >= 8
except Exception:
    pass
status = widgets.HTML(value='')
js_out = widgets.Output()  # hidden slot used only to inject the auto-download <script>
display(HTML('<style>'
             '.widget-button { border-radius: 3px !important; color: white !important; '
             'font-weight: 600 !important; }'
             '</style>'))
display(widgets.VBox([btn, status, js_out]))

def trigger_download(path, name):
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    mime = 'application/zip' if name.endswith('.zip') else (mimetypes.guess_type(name)[0] or 'application/octet-stream')
    js_out.outputs = ()
    js_out.append_display_data(HTML(
        f'<script>var a=document.createElement("a");a.download="{name}";'
        f'a.href="data:{mime};base64,{b64}";document.body.appendChild(a);a.click();a.remove();</script>'))

def plan_parts():
    """Group files into zip parts, each under CHUNK_MB. Single small file -> as-is."""
    paths = [os.path.join(OUTPUT_DIR, f) for f in results]
    if len(paths) == 1 and os.path.getsize(paths[0]) / 1e6 <= CHUNK_MB:
        return [('file', paths[0])]  # one small file: download it directly, no zip
    parts, cur, cur_mb = [], [], 0.0
    for p in paths:
        mb = os.path.getsize(p) / 1e6
        if cur and cur_mb + mb > CHUNK_MB:
            parts.append(cur); cur, cur_mb = [], 0.0
        cur.append(p); cur_mb += mb
    if cur:
        parts.append(cur)
    return [('zip', part) for part in parts]

def on_click(_):
    btn.disabled = True
    try:
        parts = plan_parts()
        if len(parts) == 1 and parts[0][0] == 'file':
            path = parts[0][1]
            name = os.path.basename(path)
            status.value = SPINNER.replace('MSG', f'Encoding {name}...')
            trigger_download(path, name)
            status.value = ok_html(f'Download of {name} started — check the browser downloads bar.')
            return
        n = len(parts)
        for i, (_, files) in enumerate(parts, 1):
            zname = 'odtsr_results.zip' if n == 1 else f'odtsr_results_part{i}of{n}.zip'
            zpath = f'/kaggle/working/{zname}'
            status.value = SPINNER.replace('MSG', f'Packing {zname} ({i}/{n})...')
            if os.path.exists(zpath):
                os.remove(zpath)
            # ZIP_STORED: PNGs are already compressed, DEFLATE would just burn CPU
            with zipfile.ZipFile(zpath, 'w', compression=zipfile.ZIP_STORED) as zf:
                for p in files:
                    zf.write(p, arcname=os.path.basename(p))
            status.value = SPINNER.replace('MSG', f'Starting download {i}/{n}: {zname}...')
            trigger_download(zpath, zname)
            if i < n:
                time.sleep(2)  # let the browser register the previous download
        status.value = ok_html(
            f'Started {n} download(s) — check the browser downloads bar.' +
            ('' if n == 1 else ' Unpack all parts into one folder.'))
    except Exception as e:
        status.value = err_html(f'Error: {e}')
    finally:
        btn.disabled = False

btn.on_click(on_click)